In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import torch

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

Every pixel in th stimulus has x,y coordinate. Every eletrode has a position in the body, map the eletrodes (x,y)
DeepRF: Ultrafast population receptive field mapping with deep learning
Jordy Thielen, Umut Güçlü, Yagmur Güçlütürk, Luca Ambrogioni, Sander E. Bosch, Marcel A. J. van Gerven
bioRxiv 732990; doi: https://doi.org/10.1101/732990

The Discrimination of Two-point Touch Sense for
the Upper Extremity in Indian Adults
Kannathu Shibin, Asir John Samuel

In [ ]:
TPD_FINGER = 2.5
TPD_HAND = 14.7
TPD_FOREARM = 27.5
TPD_ARM_LOWER = 27.6
TPD_ARM_MID = 36.0
TPD_ARM_UPPER = 36.8

GAP_FINGER_TO_HAND = TPD_HAND
GAP_HAND_TO_FOREARM = TPD_FOREARM
GAP_FOREARM_TO_ARM = TPD_ARM_LOWER

ELECTRODE_POSITIONS = {}

ELECTRODE_POSITIONS['E1'] = 0.0
ELECTRODE_POSITIONS['E2'] = TPD_FINGER
ELECTRODE_POSITIONS['E3'] = 2 * TPD_FINGER

ELECTRODE_POSITIONS['E4'] = ELECTRODE_POSITIONS['E3'] + GAP_FINGER_TO_HAND
for i, e in enumerate(['E5', 'E6', 'E7'], start=1):
    ELECTRODE_POSITIONS[e] = ELECTRODE_POSITIONS['E4'] + i * TPD_HAND

ELECTRODE_POSITIONS['E8'] = ELECTRODE_POSITIONS['E7'] + GAP_HAND_TO_FOREARM
for i, e in enumerate(['E9', 'E10', 'E11', 'E12', 'E13'], start=1):
    ELECTRODE_POSITIONS[e] = ELECTRODE_POSITIONS['E8'] + i * TPD_FOREARM

ELECTRODE_POSITIONS['E14'] = ELECTRODE_POSITIONS['E13'] + GAP_FOREARM_TO_ARM
arm_spacings = [TPD_ARM_LOWER, TPD_ARM_LOWER, TPD_ARM_MID, TPD_ARM_MID, TPD_ARM_UPPER, TPD_ARM_UPPER]
arm_electrodes = ['E15', 'E16', 'E17', 'E18', 'E19', 'E20']
cumulative = ELECTRODE_POSITIONS['E14']
for e, spacing in zip(arm_electrodes, arm_spacings):
    cumulative += spacing
    ELECTRODE_POSITIONS[e] = cumulative

SOMATOTOPIC_RANGE = (0.0, ELECTRODE_POSITIONS['E20'])

REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

print("Electrode positions (mm):")
for region, electrodes in REGION_TO_ELECTRODES.items():
    positions = [ELECTRODE_POSITIONS[e] for e in electrodes]
    print(f"\n  {region}:")
    for e in electrodes:
        print(f"    {e}: {ELECTRODE_POSITIONS[e]:7.1f} mm")
print(f"\nTotal span: {SOMATOTOPIC_RANGE[1]:.1f} mm ({SOMATOTOPIC_RANGE[1]/10:.1f} cm)")

In [ ]:
from pathlib import Path
import json

BIDS_ROOT = Path(r"../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subject = "sub-p0002"
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs = 4

bold_json = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])

RESULTS_DIR = Path("results/DeepRF")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

all_events = []
for run in range(1, n_runs + 1):
    events_path = (BIDS_ROOT / subject / session / "func" /
                   f"{subject}_{session}_{task}_run-{run}_events.tsv")
    df = pd.read_csv(events_path, sep='\t')
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()


In [ ]:
from nilearn.image import load_img, index_img

first_run_path = (DERIVATIVES / subject / session / "func" /
                  f"{subject}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)
n_volumes_per_run = first_run_img.shape[3]

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import new_img_like, resample_to_img
from nilearn.maskers import NiftiMasker

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels)
              if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')

masker = NiftiMasker(mask_img=s1_mask_resampled, standardize='zscore_sample', high_pass=0.01, t_r=tr)
masker.fit(first_run_img)

In [ ]:
from nilearn.glm.first_level.hemodynamic_models import spm_hrf

#prf parameters (mu, sigma) and stimulus sequence, produce a predicted BOLD time series via HRF convolution
def generate_predicted_bold(mu, sigma, run_events, tr, n_volumes):
    oversample = 10
    dt = tr/oversample
    n_samples_hr = int(n_volumes * tr / dt)
    neural = np.zeros(n_samples_hr)
    for _, event in run_events.iterrows():
        onset = float(event['onset'])
        duration = float(event['duration'])
        electrode = event['trial_type']
        pos = ELECTRODE_POSITIONS[electrode]
        response = np.exp(-0.5 * ((pos - mu) / max(sigma, 0.1)) ** 2)
        start_idx = int(onset / dt)
        end_idx = int((onset + duration) / dt)
        end_idx = min(end_idx, n_samples_hr)
        if start_idx < n_samples_hr:
            neural[start_idx:end_idx] = response

    hrf = spm_hrf(dt, oversampling=1)
    bold_hr = np.convolve(neural, hrf)[:n_samples_hr]
    bold = bold_hr[::oversample][:n_volumes]

    return bold

In [ ]:
import matplotlib.pyplot as plt

run1_events = stim_events[stim_events['run'] == 1]
test_bold = generate_predicted_bold(mu=2.5, sigma=5.0, run_events=run1_events,
                                     tr=tr, n_volumes=n_volumes_per_run)
plt.figure(figsize=(14, 3))
plt.plot(np.arange(len(test_bold)) * tr, test_bold)
plt.xlabel('Time (s)')
plt.ylabel('Predicted BOLD')
plt.title('Synthetic BOLD for μ=2.5mm (finger), σ=5.0mm')
plt.tight_layout()
plt.show()
print(f"Time series length: {len(test_bold)}, max: {test_bold.max():.4f}")

In [ ]:
from nilearn.signal import clean as nilearn_clean

def extract_voxel_timeseries(run):
    bold_path = (DERIVATIVES / subject / session / "func" /
                 f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
    bold_img = load_img(str(bold_path))
    return masker.transform(bold_img)

n_voxels = int(masker.mask_img_.get_fdata().sum())
all_run_ts = []
for run in range(1, n_runs + 1):
    ts_z = extract_voxel_timeseries(run)
    all_run_ts.append(ts_z)
    print(f"Run {run}: {ts_z.shape}")
print(f"\n{n_voxels} voxels in S1 mask")

## 1. Grid search pRF mapping (Classical)
Exhaustive search over a grid of (μ, σ) parameters. For each candidate, generate predicted BOLD via the forward model, and find the (μ, σ) that maximizes R² (with OLS scaling) averaged across all 4 runs.

In [ ]:
import time

mu_grid = np.linspace(-30, 490, 150)
sigma_grid = np.array([2, 5, 10, 20, 40, 80, 120, 200, 300, 500])
n_grid = len(mu_grid) * len(sigma_grid)
print(f"Grid: {len(mu_grid)} μ × {len(sigma_grid)} σ = {n_grid} models")


print("forward model grid for all runs")
pred_grids = {}
grid_params = []

for run_idx in range(n_runs):
    run_ev = stim_events[stim_events['run'] == run_idx + 1]
    pred_list = []
    if run_idx == 0:
        grid_params = []
    for mu in mu_grid:
        for sigma in sigma_grid:
            bold = generate_predicted_bold(mu, sigma, run_ev, tr, n_volumes_per_run)
            bold_f = nilearn_clean(bold.reshape(-1, 1), t_r=tr,
                                   high_pass=0.01, standardize='zscore_sample').flatten()
            pred_list.append(bold_f)
            if run_idx == 0:
                grid_params.append((mu, sigma))
    pred_grids[run_idx] = np.column_stack(pred_list)
    print(f"  Run {run_idx+1}: {pred_grids[run_idx].shape}")

grid_params = np.array(grid_params)
print(f"\nSearching {n_grid} models × {n_voxels} voxels × {n_runs} runs...")
t0 = time.time()

best_r2 = np.full(n_voxels, -np.inf)
best_grid_idx = np.zeros(n_voxels, dtype=int)

for g in range(n_grid):
    r2_sum = np.zeros(n_voxels)
    for run_idx in range(n_runs):
        pred_g = pred_grids[run_idx][:, g]
        actual = all_run_ts[run_idx]

        X = np.column_stack([pred_g, np.ones(len(pred_g))])
        betas, _, _, _ = np.linalg.lstsq(X, actual, rcond=None)
        fitted = X @ betas
        ss_res = np.sum((actual - fitted) ** 2, axis=0)
        ss_tot = np.sum((actual - actual.mean(axis=0, keepdims=True)) ** 2, axis=0)
        r2_sum += np.where(ss_tot > 1e-8, 1.0 - ss_res / ss_tot, 0.0)

    r2_avg = r2_sum / n_runs
    improved = r2_avg > best_r2
    best_r2[improved] = r2_avg[improved]
    best_grid_idx[improved] = g

    if (g + 1) % 300 == 0:
        print(f"  {g+1}/{n_grid} done...")

mu_map = grid_params[best_grid_idx, 0]
sigma_map = grid_params[best_grid_idx, 1]
r_squared = best_r2.copy()

elapsed = time.time() - t0
R2_THRESHOLD = 0.05
good_voxels = r_squared > R2_THRESHOLD

print(f"\nDone in {elapsed:.1f}s")
print(f"{'='*60}")
print(f"Grid-search pRF results ({n_voxels} voxels, {n_runs}-run avg)")
print(f"{'='*60}")
print(f"R²:  mean={r_squared.mean():.4f}, median={np.median(r_squared):.4f}, max={r_squared.max():.4f}")
print(f"Voxels R² > 0.02: {np.sum(r_squared > 0.02)}")
print(f"Voxels R² > 0.05: {np.sum(r_squared > 0.05)}")
print(f"Voxels R² > 0.10: {np.sum(r_squared > 0.10)}")
if good_voxels.sum() > 0:
    print(f"\nGood voxels (R²>{R2_THRESHOLD}): {good_voxels.sum()}")
    print(f"  μ: {mu_map[good_voxels].mean():.1f} ± {mu_map[good_voxels].std():.1f} mm")
    print(f"  σ: {sigma_map[good_voxels].mean():.1f} ± {sigma_map[good_voxels].std():.1f} mm")
    print(f"\n  σ distribution:")
    for s in sigma_grid:
        c = np.sum(sigma_map[good_voxels] == s)
        if c > 0:
            print(f"    σ={s:>3.0f}mm: {c} voxels")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(r_squared, bins=50, edgecolor='black')
plt.axvline(R2_THRESHOLD, color='r', linestyle='--', label=f'Threshold = {R2_THRESHOLD}')
plt.xlabel('R²')
plt.ylabel('Number of voxels')
plt.title(f'Grid-search pRF goodness-of-fit ({n_runs}-run average)')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'r_squared_histogram.png', dpi=150)
plt.show()

In [ ]:
top_k = 10
top_idx = np.argsort(r_squared)[-top_k:][::-1]

run1_ev = stim_events[stim_events['run'] == 1]
fig, axes = plt.subplots(top_k, 1, figsize=(14, 2.5 * top_k), sharex=True)
time_axis = np.arange(n_volumes_per_run) * tr

for i, vi in enumerate(top_idx):
    predicted = generate_predicted_bold(mu_map[vi], sigma_map[vi], run1_ev, tr, n_volumes_per_run)
    predicted_f = nilearn_clean(predicted.reshape(-1, 1), t_r=tr,
                                high_pass=0.01, standardize='zscore_sample').flatten()
    actual = all_run_ts[0][:, vi]

    X = np.column_stack([predicted_f, np.ones(len(predicted_f))])
    betas, _, _, _ = np.linalg.lstsq(X, actual, rcond=None)
    fitted = X @ betas

    # Label body region from mu
    mu_v = mu_map[vi]
    if mu_v < ELECTRODE_POSITIONS['E4']:
        region = 'Finger'
    elif mu_v < ELECTRODE_POSITIONS['E8']:
        region = 'Hand'
    elif mu_v < ELECTRODE_POSITIONS['E14']:
        region = 'Forearm'
    else:
        region = 'Arm'

    axes[i].plot(time_axis, actual, 'b-', alpha=0.7, label='Actual BOLD')
    axes[i].plot(time_axis, fitted, 'r-', alpha=0.7, label='Grid-search fit')
    axes[i].set_ylabel(f'Voxel {vi}')
    axes[i].set_title(f'R²={r_squared[vi]:.4f} | μ={mu_v:.0f}mm ({region}), σ={sigma_map[vi]:.0f}mm', fontsize=9)
    if i == 0:
        axes[i].legend(fontsize=8)

axes[-1].set_xlabel('Time (s)')
plt.suptitle(f'Top {top_k} voxels by R² (grid-search)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_voxels_timeseries.png', dpi=150)
plt.show()

## 2. DeepRF - CNN pRF estimation
Train a 1D CNN on synthetic pRF time series (Thielen et al., 2019 adapted to somatosensory). The network learns to map a BOLD time series directly to (μ, σ) in a single forward pass.

In [ ]:
def generate_training_data(n_samples, 
                           run_events, 
                           tr, 
                           n_volumes, 
                           mu_range=(-30, 490), 
                           sigma_range=(1, 500), 
                           noise_levels=(0, 0.5, 1, 2)
                           ):
    X_all, Y_all = [], []
    for i in range(n_samples):
        mu = rng.uniform(*mu_range)
        sigma = rng.uniform(*sigma_range)
        bold_clean = generate_predicted_bold(mu, sigma, run_events, tr, n_volumes)
        for nl in noise_levels:
            if nl == 0.0:
                bold = bold_clean.copy()
            else:
                bold = bold_clean + rng.normal(0, nl * (np.std(bold_clean) + 1e-8), size=bold_clean.shape)
            bold = nilearn_clean(bold.reshape(-1, 1), t_r=tr,
                                 high_pass=0.01, standardize='zscore_sample').flatten()
            X_all.append(bold)
            Y_all.append([mu, sigma])
        if (i + 1) % 5000 == 0:
            print(f"  Generated {i+1}/{n_samples} base samples...")
    return np.array(X_all, dtype=np.float32), np.array(Y_all, dtype=np.float32)

In [ ]:
run1_events = stim_events[stim_events['run'] == 1]
X_synth, Y_synth = generate_training_data(
    n_samples=10000, run_events=run1_events,
    tr=tr, n_volumes=n_volumes_per_run)
print(f"Synthetic dataset: X={X_synth.shape}, Y={Y_synth.shape}")
print(f"  {len(X_synth)} samples (10k base × 4 noise levels)")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import time

class DeepRF(nn.Module):
    def __init__(self, n_timepoints, n_outputs=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(8),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_outputs),
        )
    def forward(self, x):
        return self.regressor(self.features(x))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepRF(n_timepoints=X_synth.shape[1]).to(device)

mu_mean, mu_std = Y_synth[:, 0].mean(), Y_synth[:, 0].std()
sigma_mean, sigma_std = Y_synth[:, 1].mean(), Y_synth[:, 1].std()
Y_norm = Y_synth.copy()
Y_norm[:, 0] = (Y_norm[:, 0] - mu_mean) / mu_std
Y_norm[:, 1] = (Y_norm[:, 1] - sigma_mean) / sigma_std

X_t = torch.FloatTensor(X_synth).unsqueeze(1)
Y_t = torch.FloatTensor(Y_norm)
n_val = int(0.1 * len(X_t))
idx = torch.randperm(len(X_t), generator=torch.Generator().manual_seed(RANDOM_SEED))
val_idx, train_idx = idx[:n_val], idx[n_val:]

X_val, Y_val = X_t[val_idx].to(device), Y_t[val_idx].to(device)
train_loader = DataLoader(TensorDataset(X_t[train_idx], Y_t[train_idx]),
                          batch_size=512, shuffle=True, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, min_lr=1e-6)
criterion = nn.MSELoss()

n_epochs = 60
best_val_loss, best_state = float('inf'), None
train_losses, val_losses = [], []

In [ ]:
for epoch in range(n_epochs):
    t0 = time.time()
    model.train()
    t_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        loss = criterion(model(bx), by)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        t_loss += loss.item() * bx.size(0)
    t_loss /= len(train_loader.dataset)

    model.eval()
    with torch.no_grad():
        v_loss = criterion(model(X_val), Y_val).item()
    train_losses.append(t_loss); val_losses.append(v_loss)
    scheduler.step(v_loss)
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d}/{n_epochs} | Train: {t_loss:.4f} | Val: {v_loss:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e} | {time.time()-t0:.1f}s")

model.load_state_dict(best_state)
model = model.cpu()
torch.save(model.state_dict(), RESULTS_DIR / 'deeprf_model.pt')
print(f"\nBest validation loss: {best_val_loss:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE (normalised)')
axes[0].set_title('Training curves'); axes[0].legend(); axes[0].set_yscale('log')
model.eval()
with torch.no_grad():
    val_preds_norm = model(X_val.cpu()).numpy()
    val_true_norm = Y_val.cpu().numpy()
val_preds = val_preds_norm.copy(); val_true = val_true_norm.copy()
val_preds[:, 0] = val_preds_norm[:, 0] * mu_std + mu_mean
val_preds[:, 1] = val_preds_norm[:, 1] * sigma_std + sigma_mean
val_true[:, 0] = val_true_norm[:, 0] * mu_std + mu_mean
val_true[:, 1] = val_true_norm[:, 1] * sigma_std + sigma_mean

for i, (name, unit, ax_idx) in enumerate([('μ', 'mm', 1), ('σ', 'mm', 2)]):
    ax = axes[ax_idx]
    ax.scatter(val_true[:, i], val_preds[:, i], alpha=0.05, s=1)
    lims = [min(val_true[:, i].min(), val_preds[:, i].min()),
            max(val_true[:, i].max(), val_preds[:, i].max())]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlabel(f'True {name} ({unit})'); ax.set_ylabel(f'Predicted {name} ({unit})')
    corr = np.corrcoef(val_true[:, i], val_preds[:, i])[0, 1]
    mae = np.mean(np.abs(val_true[:, i] - val_preds[:, i]))
    ax.set_title(f'{name}: r={corr:.3f}, MAE={mae:.1f}{unit}')

axes[3].hist(np.abs(val_preds[:, 0] - val_true[:, 0]), bins=50, alpha=0.6, label='|Δμ|')
axes[3].hist(np.abs(val_preds[:, 1] - val_true[:, 1]), bins=50, alpha=0.6, label='|Δσ|')
axes[3].set_xlabel('Absolute error (mm)'); axes[3].set_title('Error distribution'); axes[3].legend()

plt.suptitle('DeepRF CNN — Synthetic validation', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cnn_validation.png', dpi=150)
plt.show()

In [ ]:
model.eval()
all_run_params_cnn = []
for run_idx in range(n_runs):
    voxel_input = torch.FloatTensor(all_run_ts[run_idx].T).unsqueeze(1)
    with torch.no_grad():
        preds_norm = []
        for i in range(0, voxel_input.shape[0], 500):
            preds_norm.append(model(voxel_input[i:i+500]).numpy())
        preds_norm = np.concatenate(preds_norm, axis=0)
    preds = preds_norm.copy()
    preds[:, 0] = preds_norm[:, 0] * mu_std + mu_mean
    preds[:, 1] = preds_norm[:, 1] * sigma_std + sigma_mean
    all_run_params_cnn.append(preds)

cnn_params_avg = np.mean(all_run_params_cnn, axis=0)
mu_map_cnn = cnn_params_avg[:, 0]
sigma_map_cnn = np.clip(cnn_params_avg[:, 1], 0.1, None)

cnn_r2_runs = []
for run_idx in range(n_runs):
    run_ev = stim_events[stim_events['run'] == run_idx + 1]
    r2 = np.zeros(n_voxels)
    for v in range(n_voxels):
        pred = generate_predicted_bold(mu_map_cnn[v], sigma_map_cnn[v], run_ev, tr, n_volumes_per_run)
        pred_f = nilearn_clean(pred.reshape(-1, 1), t_r=tr,
                               high_pass=0.01, standardize='zscore_sample').flatten()
        if np.std(pred_f) < 1e-8:
            continue
        actual_v = all_run_ts[run_idx][:, v]
        X = np.column_stack([pred_f, np.ones(len(pred_f))])
        betas, _, _, _ = np.linalg.lstsq(X, actual_v, rcond=None)
        fitted = X @ betas
        ss_res = np.sum((actual_v - fitted) ** 2)
        ss_tot = np.sum((actual_v - actual_v.mean()) ** 2)
        r2[v] = 1.0 - ss_res / ss_tot if ss_tot > 1e-8 else 0.0
    cnn_r2_runs.append(r2)
    print(f"CNN Run {run_idx+1}: mean R²={r2.mean():.4f}, max={r2.max():.4f}")

r_squared_cnn = np.mean(cnn_r2_runs, axis=0)
print(f"\nCNN R² (4-run avg): mean={r_squared_cnn.mean():.4f}, max={r_squared_cnn.max():.4f}")
print(f"CNN voxels R²>0.02: {np.sum(r_squared_cnn > 0.02)}")
print(f"CNN voxels R²>0.05: {np.sum(r_squared_cnn > 0.05)}")

## Comparison: Grid-search vs CNN
Side-by-side comparison of both pRF mapping approaches on the same data.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# R² histograms
axes[0, 0].hist(r_squared, bins=50, alpha=0.7, label='Grid-search', color='steelblue', edgecolor='black')
axes[0, 0].hist(r_squared_cnn, bins=50, alpha=0.7, label='CNN', color='coral', edgecolor='black')
axes[0, 0].axvline(0.02, color='grey', ls='--', lw=1)
axes[0, 0].set_xlabel('R²'); axes[0, 0].set_ylabel('Voxel count')
axes[0, 0].set_title('R² distribution'); axes[0, 0].legend()

# R² scatter (voxel-wise)
axes[0, 1].scatter(r_squared, r_squared_cnn, alpha=0.1, s=3)
lim = max(r_squared.max(), r_squared_cnn.max()) * 1.1
axes[0, 1].plot([0, lim], [0, lim], 'r--', lw=1)
axes[0, 1].set_xlabel('Grid-search R²'); axes[0, 1].set_ylabel('CNN R²')
axes[0, 1].set_title('Voxel-wise R² comparison')

# μ comparison (for well-fit voxels)
mask_both = (r_squared > 0.02) & (r_squared_cnn > 0.005)
if mask_both.sum() > 5:
    axes[0, 2].scatter(mu_map[mask_both], mu_map_cnn[mask_both], alpha=0.5, s=10)
    axes[0, 2].plot([0, 460], [0, 460], 'r--', lw=1)
    axes[0, 2].set_xlabel('Grid μ (mm)'); axes[0, 2].set_ylabel('CNN μ (mm)')
    r_mu = np.corrcoef(mu_map[mask_both], mu_map_cnn[mask_both])[0, 1]
    axes[0, 2].set_title(f'μ agreement (r={r_mu:.3f}, n={mask_both.sum()})')
else:
    axes[0, 2].text(0.5, 0.5, 'Too few overlapping\nvoxels', ha='center', va='center',
                    transform=axes[0, 2].transAxes, fontsize=12)
    axes[0, 2].set_title('μ agreement')

# σ comparison
if mask_both.sum() > 5:
    axes[1, 0].scatter(sigma_map[mask_both], sigma_map_cnn[mask_both], alpha=0.5, s=10)
    axes[1, 0].plot([0, 500], [0, 500], 'r--', lw=1)
    axes[1, 0].set_xlabel('Grid σ (mm)'); axes[1, 0].set_ylabel('CNN σ (mm)')
    axes[1, 0].set_title('σ agreement')

# Somatotopic distribution comparison
regions = list(REGION_TO_ELECTRODES.keys())
grid_counts, cnn_counts = [], []
for region, electrodes in REGION_TO_ELECTRODES.items():
    lo = ELECTRODE_POSITIONS[electrodes[0]]
    hi = ELECTRODE_POSITIONS[electrodes[-1]]
    if region == 'Middle_Finger':
        lo = -30
    grid_c = np.sum((mu_map[r_squared > 0.02] >= lo) & (mu_map[r_squared > 0.02] <= hi))
    cnn_c = np.sum((mu_map_cnn[r_squared_cnn > 0.02] >= lo) & (mu_map_cnn[r_squared_cnn > 0.02] <= hi))
    grid_counts.append(grid_c); cnn_counts.append(cnn_c)

x = np.arange(len(regions))
axes[1, 1].bar(x - 0.2, grid_counts, 0.4, label='Grid-search', color='steelblue')
axes[1, 1].bar(x + 0.2, cnn_counts, 0.4, label='CNN', color='coral')
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(regions, rotation=15)
axes[1, 1].set_ylabel('Voxel count (R²>0.02)'); axes[1, 1].set_title('Somatotopic distribution')
axes[1, 1].legend()

# Summary table
summary = (
    f"{'Metric':<25} {'Grid-search':>12} {'CNN':>12}\n"
    f"{'─'*50}\n"
    f"{'Mean R²':<25} {r_squared.mean():>12.4f} {r_squared_cnn.mean():>12.4f}\n"
    f"{'Max R²':<25} {r_squared.max():>12.4f} {r_squared_cnn.max():>12.4f}\n"
    f"{'Voxels R²>0.02':<25} {np.sum(r_squared>0.02):>12d} {np.sum(r_squared_cnn>0.02):>12d}\n"
    f"{'Voxels R²>0.05':<25} {np.sum(r_squared>0.05):>12d} {np.sum(r_squared_cnn>0.05):>12d}\n"
    f"{'Mean μ (good vox)':<25} {mu_map[r_squared>0.02].mean():>12.1f} {mu_map_cnn[r_squared_cnn>0.02].mean() if np.sum(r_squared_cnn>0.02)>0 else float('nan'):>12.1f}\n"
    f"{'Mean σ (good vox)':<25} {sigma_map[r_squared>0.02].mean():>12.1f} {sigma_map_cnn[r_squared_cnn>0.02].mean() if np.sum(r_squared_cnn>0.02)>0 else float('nan'):>12.1f}\n"
)
axes[1, 2].text(0.05, 0.95, summary, transform=axes[1, 2].transAxes, fontsize=10,
                va='top', ha='left', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[1, 2].set_title('Summary'); axes[1, 2].axis('off')

plt.suptitle('Grid-search vs CNN pRF mapping — sub-p0002, L S1', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'grid_vs_cnn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(summary)

In [ ]:
from nilearn.plotting import plot_stat_map, plot_glass_brain

R2_PLOT_THRESHOLD = 0.02

# Grid-search maps
mu_grid_masked = np.where(r_squared > R2_PLOT_THRESHOLD, mu_map, 0)
r2_grid_masked = np.where(r_squared > R2_PLOT_THRESHOLD, r_squared, 0)
mu_grid_img = masker.inverse_transform(mu_grid_masked.reshape(1, -1))
r2_grid_img = masker.inverse_transform(r2_grid_masked.reshape(1, -1))

# CNN maps
mu_cnn_masked = np.where(r_squared_cnn > R2_PLOT_THRESHOLD, mu_map_cnn, 0)
r2_cnn_masked = np.where(r_squared_cnn > R2_PLOT_THRESHOLD, r_squared_cnn, 0)
mu_cnn_img = masker.inverse_transform(mu_cnn_masked.reshape(1, -1))
r2_cnn_img = masker.inverse_transform(r2_cnn_masked.reshape(1, -1))

cuts = np.arange(30, 72, 6)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

plot_stat_map(mu_grid_img, bg_img=ref_img, cmap='hot', colorbar=True, axes=axes[0, 0],
              title=f'Grid-search: μ (mm) — {np.sum(r_squared>R2_PLOT_THRESHOLD)} voxels',
              display_mode='z', cut_coords=cuts)
plot_stat_map(r2_grid_img, bg_img=ref_img, cmap='inferno', colorbar=True, axes=axes[0, 1],
              title='Grid-search: R²', display_mode='z', cut_coords=cuts)
plot_stat_map(mu_cnn_img, bg_img=ref_img, cmap='hot', colorbar=True, axes=axes[1, 0],
              title=f'CNN: μ (mm) — {np.sum(r_squared_cnn>R2_PLOT_THRESHOLD)} voxels',
              display_mode='z', cut_coords=cuts)
plot_stat_map(r2_cnn_img, bg_img=ref_img, cmap='inferno', colorbar=True, axes=axes[1, 1],
              title='CNN: R²', display_mode='z', cut_coords=cuts)

plt.savefig(FIGURES_DIR / 'brain_maps_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top_k = 5
top_grid = np.argsort(r_squared)[-top_k:][::-1]
run1_ev = stim_events[stim_events['run'] == 1]

fig, axes = plt.subplots(top_k, 2, figsize=(20, 3 * top_k), sharex=True)
time_axis = np.arange(n_volumes_per_run) * tr

for i, vi in enumerate(top_grid):
    actual = all_run_ts[0][:, vi]

    # Grid fit
    pred_g = generate_predicted_bold(mu_map[vi], sigma_map[vi], run1_ev, tr, n_volumes_per_run)
    pred_gf = nilearn_clean(pred_g.reshape(-1, 1), t_r=tr, high_pass=0.01, standardize='zscore_sample').flatten()
    X = np.column_stack([pred_gf, np.ones(len(pred_gf))])
    betas, _, _, _ = np.linalg.lstsq(X, actual, rcond=None)
    fitted_g = X @ betas

    # CNN fit
    pred_c = generate_predicted_bold(mu_map_cnn[vi], sigma_map_cnn[vi], run1_ev, tr, n_volumes_per_run)
    pred_cf = nilearn_clean(pred_c.reshape(-1, 1), t_r=tr, high_pass=0.01, standardize='zscore_sample').flatten()
    X = np.column_stack([pred_cf, np.ones(len(pred_cf))])
    betas, _, _, _ = np.linalg.lstsq(X, actual, rcond=None)
    fitted_c = X @ betas

    axes[i, 0].plot(time_axis, actual, 'b-', alpha=0.7, label='Actual')
    axes[i, 0].plot(time_axis, fitted_g, 'r-', alpha=0.7, label='Grid fit')
    axes[i, 0].set_title(f'Grid: voxel {vi} | R²={r_squared[vi]:.4f} | μ={mu_map[vi]:.0f}, σ={sigma_map[vi]:.0f}', fontsize=9)

    axes[i, 1].plot(time_axis, actual, 'b-', alpha=0.7, label='Actual')
    axes[i, 1].plot(time_axis, fitted_c, 'orange', alpha=0.7, label='CNN fit')
    axes[i, 1].set_title(f'CNN: voxel {vi} | R²={r_squared_cnn[vi]:.4f} | μ={mu_map_cnn[vi]:.0f}, σ={sigma_map_cnn[vi]:.0f}', fontsize=9)

    if i == 0:
        axes[i, 0].legend(fontsize=8); axes[i, 1].legend(fontsize=8)

axes[-1, 0].set_xlabel('Time (s)'); axes[-1, 1].set_xlabel('Time (s)')
plt.suptitle('Top 5 grid-search voxels: Grid-search vs CNN fits', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_voxels_comparison.png', dpi=150)
plt.show()

In [ ]:
import nibabel as nib

np.savez(RESULTS_DIR / 'prf_results.npz',
         mu_map_grid=mu_map, sigma_map_grid=sigma_map, r_squared_grid=r_squared,
         mu_map_cnn=mu_map_cnn, sigma_map_cnn=sigma_map_cnn, r_squared_cnn=r_squared_cnn,
         best_grid_idx=best_grid_idx, grid_params=grid_params)

nib.save(mu_grid_img, RESULTS_DIR / 'mu_map_grid.nii.gz')
nib.save(r2_grid_img, RESULTS_DIR / 'r2_map_grid.nii.gz')
nib.save(mu_cnn_img, RESULTS_DIR / 'mu_map_cnn.nii.gz')
nib.save(r2_cnn_img, RESULTS_DIR / 'r2_map_cnn.nii.gz')

print(f"Results saved to {RESULTS_DIR}")